In [ ]:
"""BigAlpha 2026 factor submission — 自由代码版(不使用受限表达式语言)。

设计说明:
  比赛规则原文并未要求因子计算逻辑必须编译成SQL或限定在某个受限算子集合内,
  只要求 main(datasources, start_date, end_date) 返回 date/instrument/factor 三列、
  不联网、3小时内跑完。这里改用"SQL只做取数,因子计算用自由pandas代码"的思路:

  1. 用 dai.query() 从三张平台表分别取数:
       - bar1m: 聚合成日频 OHLCV(这一步SQL聚合是必须的,因为原始数据是分钟级)
       - bigalpha_2026_factorlib: 直接拿现成的估值/技术指标因子(pe_ttm/pb/rsi等)
       - bigalpha_2026_exposure: 拿行业代码 + 用于中证1000 point-in-time 成分股筛选
  2. 把平台字段名"翻译"回我们挖掘时用的本地字段名(PE/PB/market_value/citic1_code等),
     这样挖出来的原始因子函数(factor_main)可以【原封不动】直接调用,不需要改因子代码本身,
     降低了"改代码引入新bug"的风险。
  3. 直接调用 factor_main(df) 算出因子值,按中证1000成分股筛选,返回标准三列。

AI-track disclosure:
    因子函数由 AI FactorMiner 工作流生成并经样本内/跨年验证筛选;
    本 main() 内的取数与拼装逻辑为确定性代码,评估时不调用任何模型或外部网络。
"""

import numpy as np
import pandas as pd
import dai


# ============================================================
# 1. 在这里粘贴你要提交的那一个因子函数(原封不动,不用改字段名)
#    函数名必须叫 factor_main,只能有一个
# ============================================================

def factor_main(df):
    g = df.groupby('instrument', sort=False)

    # 成交额占流通市值比例：amount / free-float market value
    # 使用 neg_market_value 作为流通市值近似（字段名中已包含流通市值语义）
    base = df['neg_market_value'].replace(0, np.nan)
    ratio = df['amount'] / base

    # 用过去5日均值平滑，体现资金关注度的持续性
    factor = g['amount'].transform(lambda s: s.rolling(5, min_periods=3).mean()) / g['neg_market_value'].transform(lambda s: s.replace(0, np.nan))

    # 备用：若 rolling 结果过多缺失，则退化为当日比值
    factor = factor.where(factor.notna(), ratio)

    return factor


# 因子在训练/样本内数据上的带符号平均IC,用于自动判断是否需要反转方向
TRAIN_IC = -0.0747072431333805

BUFFER_DAYS = 120  # 给滚动窗口留足历史缓冲(因子里最大用到60日窗口的话,120天足够安全)


def main(datasources, start_date, end_date):
    """取数(SQL) + 自由pandas计算因子 + 返回标准三列。"""

    bar1m = datasources.get("bar1m", "bigalpha_2026_stock_bar1m")

    eval_start = pd.to_datetime(start_date).normalize()
    eval_end = pd.to_datetime(end_date).normalize()
    query_start = eval_start - pd.Timedelta(days=BUFFER_DAYS)
    query_start_str = query_start.strftime("%Y-%m-%d %H:%M:%S")
    query_end_str = pd.to_datetime(end_date).strftime("%Y-%m-%d %H:%M:%S")

    # ---- 1a. 日频 OHLCV(从分钟表聚合,这一步SQL聚合是必须的) ----
    daily_sql = f"""
    SELECT
        date::DATE::DATETIME AS date,
        instrument::STRING AS instrument,
        FIRST(open ORDER BY date) AS open,
        MAX(high) AS high,
        MIN(low) AS low,
        LAST(close ORDER BY date) AS close,
        FIRST(pre_close ORDER BY date) AS preclose,
        MAX(volume) AS volume,
        SUM(amount) AS amount
    FROM {bar1m}
    WHERE open > 0 AND high > 0 AND low > 0 AND close > 0
    GROUP BY date::DATE, instrument
    ORDER BY instrument, date
    """
    daily_df = dai.query(
        daily_sql, filters={"date": [query_start_str, query_end_str]}, compression=True,
    ).df()
    daily_df["date"] = pd.to_datetime(daily_df["date"]).dt.normalize()
    daily_df["instrument"] = daily_df["instrument"].astype("string")

    # ---- 1b. 因子库(估值+技术指标,已经是日频,直接按需要的字段取) ----
    # 按需增减字段: 如果你的因子函数不需要某些字段,可以从SELECT里去掉减少取数量
    factorlib_sql = """
    SELECT
        date, instrument,
        pe_ttm, pb, total_market_cap, float_market_cap, turn,
        roe_avg_ttm, roa_avg_ttm
    FROM bigalpha_2026_factorlib
    """
    factorlib_df = dai.query(
        factorlib_sql, filters={"date": [query_start_str, query_end_str]}, compression=True,
    ).df()
    factorlib_df["date"] = pd.to_datetime(factorlib_df["date"]).dt.normalize()
    factorlib_df["instrument"] = factorlib_df["instrument"].astype("string")

    # ---- 1c. 风险暴露表(只取行业代码,不做中证1000筛选) ----
    # 说明: 比赛规则只说"评分时按中证1000口径"，没有要求main()自己筛选股票池。
    # 平台大概率会在评分阶段自己按中证1000对齐/取交集，所以这里不做inner join筛选，
    # 直接返回全市场因子值更安全——万一自己筛选的JOIN逻辑有问题（比如exposure表
    # 某些交易日缺数据），会导致"漏掉本该有的股票-交易日组合"，触发缺失率/覆盖度校验失败；
    # 而"多返回非成分股"大概率不会被扣分，风险不对称，所以选择不筛选这条更安全的路。
    exposure_sql = """
    SELECT date, instrument, industry_level1_code
    FROM bigalpha_2026_exposure
    """
    exposure_df = dai.query(
        exposure_sql, filters={"date": [start_date, end_date]}, compression=True,
    ).df()
    exposure_df["date"] = pd.to_datetime(exposure_df["date"]).dt.normalize()
    exposure_df["instrument"] = exposure_df["instrument"].astype("string")

    # ---- 2. 拼成一张面板,并把字段名翻译回挖掘时用的本地命名 ----
    df = daily_df.merge(factorlib_df, on=["date", "instrument"], how="left")
    df = df.merge(exposure_df, on=["date", "instrument"], how="left")  # left join,不筛选股票池

    df = df.rename(columns={
        "pe_ttm": "PE", "pb": "PB",
        "total_market_cap": "market_value", "float_market_cap": "neg_market_value",
        "industry_level1_code": "citic1_code",
    })
    df["trade_date"] = df["date"]  # 部分因子代码用的是trade_date这个列名,加个别名兼容

    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)

    # ---- 3. 调用因子函数 ----
    factor_series = factor_main(df)
    direction = 1.0 if TRAIN_IC >= 0 else -1.0

    result = pd.DataFrame({
        "date": df["date"].values,
        "instrument": df["instrument"].values,
        "factor": (direction * factor_series).values,
    })
    result["factor"] = pd.to_numeric(result["factor"], errors="coerce").replace([np.inf, -np.inf], np.nan)

    result = result[(result["date"] >= eval_start) & (result["date"] <= eval_end)]
    result = result.sort_values(["date", "instrument"], ignore_index=True)

    if list(result.columns) != ["date", "instrument", "factor"]:
        raise RuntimeError("Invalid submission columns")
    if result.empty:
        raise RuntimeError("Factor output is empty")
    return result
